# 02 – Modeling

Train model, **ROC/KS**, and **SHAP summary plot**. Runs top-to-bottom for a hiring-manager skim.

In [ ]:
# Project root and imports
import sys
from pathlib import Path
ROOT = Path().resolve().parent if Path().resolve().name == "notebooks" else Path().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from src.config import DATA_DIR, RANDOM_STATE
from src.data.schema import CORE_FEATURES, TARGET_FRAUD
from src.data.loaders import generate_and_save
from src.evaluation.metrics import compute_auc_ks
from src.models.gbm import GBMRiskModel

%matplotlib inline
DATA_PATH = ROOT / "data" / "training_data.csv"

In [ ]:
# Load data (generate if missing)
if not DATA_PATH.exists():
    df = generate_and_save(path=DATA_PATH, n=25_000, fraud_rate=0.05, random_state=RANDOM_STATE)
else:
    df = pd.read_csv(DATA_PATH)
X = df[CORE_FEATURES]
y = df[TARGET_FRAUD]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
print(f"Train: {len(X_train):,}  Test: {len(X_test):,}")

In [ ]:
# Train GBM (fast SHAP with TreeExplainer)
model = GBMRiskModel(random_state=RANDOM_STATE)
model.fit(X_train, y_train)
y_proba = model.predict_proba(X_test)[:, 1]
auc, ks, fpr, tpr, _ = compute_auc_ks(y_test.values, y_proba)
print(f"AUC: {auc:.4f}  KS: {ks:.4f}")

## ROC curve and KS

ROC curve with AUC; KS = max |TPR - FPR|.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC (AUC = {auc:.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC curve (KS = {:.3f})".format(ks))
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## SHAP summary plot

Feature impact on model output (beeswarm). Sample for speed.

In [ ]:
import shap

# TreeExplainer for GBM (fast)
explainer = shap.TreeExplainer(
    model._estimator.booster_,
    feature_perturbation="tree_path_dependent",
)
# Use a sample for quick summary plot (e.g. 500 rows)
sample_size = min(500, len(X_test))
X_sample = X_test.iloc[:sample_size]
shap_values = explainer.shap_values(X_sample)
if isinstance(shap_values, list):
    shap_values = shap_values[1]  # positive class

shap.summary_plot(shap_values, X_sample, show=False)
plt.title("SHAP summary (impact on fraud probability)")
plt.tight_layout()
plt.show()